# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup + `assemble(D)` — one frame builder, any decision date

`assemble(D)` generalises the ML-08 assembly (`cell-pre-4` + the label-quality gate + the feature build) to any monthly decision date. Feature window `[D-90d, D)`, label window `[D, D+1 month)`. It applies the same eligibility as ML-08 **plus the formalised label-quality gate** (keep only clients that still report in the label window), and reconciles the `D = 2026-03-01` frame to ML-08's 58,583 rows / base rate 0.045.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
from pathlib import Path

# Token order: env var -> .env file (local runs) -> Colab Secret -> prompt (last resort).
# Never commit the token: `.env` is gitignored and this repo is public.
def _from_dotenv(key):
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / ".env"
        if f.is_file():
            for line in f.read_text().splitlines():
                s = line.strip()
                if s.startswith(f"{key}=") or s.startswith(f"export {key}="):
                    return s.split("=", 1)[1].strip().strip('\"').strip("'")
    return None

HF_TOKEN = os.environ.get("HF_TOKEN") or _from_dotenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "no valid HF READ token found (env / .env / Colab secret)"

In [3]:
import duckdb, json
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
SEED = 42
K_ABS = 50

def daily(months):
    """read_parquet over an explicit list of month partitions (hf:// has no brace globs)."""
    paths = [f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in months]
    return f"read_parquet([{', '.join(paths)}])"

# --- decision date -> (feature month partitions, label month partition) ---
FEAT_MAP = {
    '2026-03-01': (['2025-12', '2026-01', '2026-02'], '2026-03'),
    '2026-04-01': (['2026-01', '2026-02', '2026-03'], '2026-04'),
    '2026-05-01': (['2026-02', '2026-03', '2026-04'], '2026-05'),
    '2026-06-01': (['2026-03', '2026-04', '2026-05'], '2026-06'),   # 2026-06 label = the ONE sealed read
}

# --- models + metrics: verbatim from w05_model.ipynb ---
def make_models():
    return {
        'dummy_prior': DummyClassifier(strategy='prior'),
        'logistic_regression': Pipeline([('scaler', StandardScaler()),
            ('m', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))]),
        'random_forest': RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
    }

def pct_rank(s):
    return s.rank(pct=True, method='average')

def components_from(df):
    p = (df['imp_90d'] / 3.0).replace(0, np.nan)
    pg = df['pos_last30'] - df['pos_prev60']
    return pd.DataFrame({
        'traffic_softening': pct_rank((1 - df['imp_last30'] / p).clip(0, 1).fillna(0)),
        'position_slip':     pct_rank(pg.clip(0, 20).fillna(0)) * pg.notna().astype(int),
        'reach_thinning':    pct_rank((1 - df['days_impr_last30'] / df['days_impr_prev30'].replace(0, np.nan)).clip(0, 1).fillna(0)),
    }, index=df.index)

WEIGHTS = {'traffic_softening': 0.40, 'position_slip': 0.30, 'reach_thinning': 0.30}   # frozen in ML-07

def frozen_baseline_score(df):
    c = components_from(df)
    return (sum(WEIGHTS[k] * c[k] for k in WEIGHTS) + 1e-9 * pct_rank(np.log1p(df['imp_90d']))).to_numpy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    return float(np.asarray(labels)[order[:k]].mean())

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    yy = np.asarray(labels)[order]
    if yy.sum() == 0: return 0.0
    prec = np.cumsum(yy) / (np.arange(len(yy)) + 1)
    return float((prec * yy).sum() / yy.sum())

def per_client_p_at(df, score_col, k=10):
    vals = [precision_at_k(g[score_col], g['label_decline'], k)
            for _, g in df.groupby('client_hash_id', observed=True) if len(g) >= k]
    return float(np.mean(vals)) if vals else float('nan')

NUM = ['imp_90d', 'log_imp_90d', 'clk_90d', 'ctr_90d', 'imp_last30', 'imp_prev60', 'clk_last30',
       'softening_ratio', 'pos_last30', 'pos_prev60', 'pos_gap',
       'days_impr_90d', 'days_impr_last30', 'days_impr_prev30', 'reach_ratio',
       'content_age_days', 'days_since_update', 'word_count', 'char_count',
       'search_volume', 'competition', 'category_count', 'backlinks',
       'has_word_count', 'has_search_volume', 'has_update_date', 'has_position']
CAT = ['content_type', 'main_intent', 'competition_level']
MOMENTUM = ['softening_ratio', 'reach_ratio', 'imp_last30', 'days_impr_last30', 'days_impr_prev30', 'pos_gap']
BANNED_SUBSTR = ['trend_direction', 'trend_pct', 'is_declining', 'imp_label', 'label_decline',
                 'last_optimized', 'ga4_', 'query', 'impressions_90d', 'health_score',
                 'priority_score', 'action_type']
print('setup ok. sklearn', sklearn.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)

setup ok. sklearn 1.9.0 | numpy 2.5.1 | pandas 3.0.5


In [4]:
def assemble(D, *, reconcile=None, extra_label_col=False):
    fmonths, lmonth = FEAT_MAP[D]
    Dts = pd.Timestamp(D)
    raw = con.sql(f"""
        WITH feat AS (
            SELECT f.client_hash_id, f.content_hash_id,
                   SUM(f.gsc_impressions) AS imp_90d,
                   SUM(f.gsc_clicks)      AS clk_90d,
                   SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_last30,
                   SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_prev60,
                   SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_clicks       ELSE 0 END) AS clk_last30,
                   SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_clicks       ELSE 0 END) AS clk_prev60,
                   SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_last30,
                   SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_prev60,
                   COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS days_impr_90d,
                   COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                         AND f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.report_date END) AS days_impr_last30,
                   COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                         AND f.report_date <  DATE '{D}' - INTERVAL 30 DAY
                         AND f.report_date >= DATE '{D}' - INTERVAL 60 DAY THEN f.report_date END) AS days_impr_prev30
            FROM {daily(fmonths)} f
            WHERE f.report_date >= DATE '{D}' - INTERVAL 90 DAY AND f.report_date < DATE '{D}'
            GROUP BY 1, 2
        ),
        lab AS (
            SELECT content_hash_id,
                   SUM(gsc_impressions) AS imp_label,
                   SUM(CASE WHEN report_date <  DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h1,
                   SUM(CASE WHEN report_date >= DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h2
            FROM {daily([lmonth])}
            WHERE report_date >= DATE '{D}' AND report_date < DATE '{D}' + INTERVAL 1 MONTH
            GROUP BY 1
        ),
        lab_client AS (
            SELECT client_hash_id, COUNT(*) AS client_label_rows
            FROM {daily([lmonth])}
            WHERE report_date >= DATE '{D}' AND report_date < DATE '{D}' + INTERVAL 1 MONTH
            GROUP BY 1
        ),
        lab_content AS (
            SELECT DISTINCT content_hash_id FROM {daily([lmonth])}
            WHERE report_date >= DATE '{D}' AND report_date < DATE '{D}' + INTERVAL 1 MONTH
        )
        SELECT feat.*,
               dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
               dc.search_volume, dc.competition, dc.competition_level, dc.category_count, dc.backlinks,
               dc.content_created_date, dc.content_updated_date,
               date_diff('day', dc.content_created_date, DATE '{D}') AS content_age_days,
               CASE WHEN dc.content_updated_date < DATE '{D}'
                    THEN date_diff('day', dc.content_updated_date, DATE '{D}') END AS days_since_update,
               cl.gsc_data_start,
               COALESCE(lc.client_label_rows, 0) AS client_label_rows,
               (lct.content_hash_id IS NOT NULL) AS content_in_label,
               feat.imp_90d / 3.0 AS pace_30d,
               CASE WHEN feat.imp_last30 > 0 THEN feat.sumpos_last30::DOUBLE / feat.imp_last30 END AS pos_last30,
               CASE WHEN feat.imp_prev60 > 0 THEN feat.sumpos_prev60::DOUBLE / feat.imp_prev60 END AS pos_prev60,
               COALESCE(lab.imp_label, 0)    AS imp_label,
               COALESCE(lab.imp_label_h1, 0) AS imp_label_h1,
               COALESCE(lab.imp_label_h2, 0) AS imp_label_h2
        FROM feat
        LEFT JOIN {DIM_CONTENT} dc ON feat.content_hash_id = dc.content_hash_id
        LEFT JOIN {DIM_CLIENTS} cl ON feat.client_hash_id = cl.client_hash_id
        LEFT JOIN lab              ON feat.content_hash_id = lab.content_hash_id
        LEFT JOIN lab_client lc    ON feat.client_hash_id = lc.client_hash_id
        LEFT JOIN lab_content lct  ON feat.content_hash_id = lct.content_hash_id
    """).df()

    raw['pass_volume_floor']   = raw['imp_last30'] >= 100
    raw['pass_client_history'] = raw['gsc_data_start'] <= (Dts - pd.Timedelta(days=90))
    raw['pass_not_freefall']   = ~((raw['imp_prev60'] > 0) & (raw['imp_last30'] < 0.5 * raw['imp_prev60']))
    raw['pass_client_reports'] = raw['client_label_rows'] > 0          # ML-08 label-quality gate, formalised
    raw['label_decline'] = ((raw['imp_label'] < 0.75 * raw['pace_30d']) &
                            (raw['imp_label_h2'] <= raw['imp_label_h1'])).astype(int)

    base_elig = raw['pass_volume_floor'] & raw['pass_client_history'] & raw['pass_not_freefall']
    keep = base_elig & raw['pass_client_reports']
    n_page_absent = int((base_elig & raw['pass_client_reports'] & ~raw['content_in_label']).sum())
    e = (raw[keep].sort_values(['client_hash_id', 'content_hash_id'])   # deterministic row order -> reproducible precision@K
                  .reset_index(drop=True).copy())

    # feature build: verbatim from w05_model.ipynb cell-2
    e['log_imp_90d']     = np.log1p(e['imp_90d'])
    e['ctr_90d']         = e['clk_90d'] / e['imp_90d'].replace(0, np.nan)
    e['softening_ratio'] = e['imp_last30'] / e['pace_30d'].replace(0, np.nan)
    e['pos_gap']         = e['pos_last30'] - e['pos_prev60']
    e['reach_ratio']     = e['days_impr_last30'] / e['days_impr_prev30'].replace(0, np.nan)
    e['has_word_count']    = e['word_count'].notna().astype(int)
    e['has_search_volume'] = e['search_volume'].notna().astype(int)
    e['has_update_date']   = e['days_since_update'].notna().astype(int)
    e['has_position']      = e['pos_last30'].notna().astype(int)

    X_num = e[NUM].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_cat = pd.get_dummies(e[CAT].astype('string').fillna('unknown'), prefix=CAT, dummy_na=False, dtype=float)
    X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
    y = e['label_decline'].to_numpy()
    groups = e['client_hash_id'].to_numpy()

    bad = [c for c in X.columns if any(b in c for b in BANNED_SUBSTR)]
    assert not bad, f'banned column leaked into X for {D}: {bad}'

    base = float(y.mean())
    print(f"assemble({D}): {len(e):>7,} rows | {len(np.unique(groups)):>3} clients | base rate {base:.4f} | "
          f"{n_page_absent:,} eligible pages absent from the label month (diagnostic, not filtered)")
    if reconcile is not None:
        assert abs(len(e) - reconcile[0]) <= 200 and abs(base - reconcile[1]) < 0.008, \
            f"reconcile failed for {D}: {len(e)} rows / base {base:.4f}"
        print(f"  -> reconciles with ML-08 ({reconcile[0]:,} rows, base rate {reconcile[1]})")
    return e, X, y, groups

e_mar, X_mar, y_mar, g_mar = assemble('2026-03-01', reconcile=(58_583, 0.045))

assemble(2026-03-01):  58,583 rows |  23 clients | base rate 0.0450 | 0 eligible pages absent from the label month (diagnostic, not filtered)
  -> reconciles with ML-08 (58,583 rows, base rate 0.045)


## 1. Two paper findings + my methodology questions

From `docs/paper-examples/index.html` (Door 1, the methodological version). **This is a worked example on a different label** — its `label_declined = (May clicks < 0.8 × April clicks)`, dev base rate 0.416, sealed-June base rate 0.655 — so its numbers are **not comparable** to this lane's impressions label (base rate 0.045). Two findings, questioned constructively with `writing-honest-claims`'s three checks (label source / validation design / would it survive a grouped-or-time split).

### Finding A — the sealed-June ranker (§6.4): model precision@50 = 0.740 vs June base rate 0.655

- **Where the label comes from:** clicks in the month strictly after the feature window — an observed outcome, well quarantined. Good.
- **What the validation actually tests:** on the *one* sealed month, the model's 0.740 is **below the paper's own naive "already-dipping" list (0.780)**, and only **+0.085** above the June base rate; the June base rate itself shifted **+0.239** from dev (0.416 → 0.655), which trips the paper's own proposed 0.416 ± 0.10 outcome-drift alarm. Per-fold precision@50 ranges 0.16–0.70.
- **Would it survive?** The paper is admirably explicit — §9: *"this five-fold sample does not establish reliable superiority"* and *"one walk-forward month and one sealed month; a longer multi-window backtest is the highest-value missing experiment."* **My methodology question:** the TL;DR frames this as *"a decision-support ranker for that queue,"* but on the only blind month it loses to a one-line heuristic. *How to make it stronger:* exactly the multi-window backtest the paper names — several sealed months, or several walk-forward decision dates, before the ranker is described as better than the rule.

### Finding B — the frozen-rule audit (§4): pages running 3.3–7× their first feature month declined at 0.689 vs 0.416 base

- **Where the label comes from:** the same clicks rule — but eligibility floors clicks at only **≥10 in the last feature month**, and the label is a bare `clicks < 0.8 × prior` **with no persistence check**. A page going 10 → 7 clicks is "declined" on what is essentially sampling noise.
- **What the finding tests:** a real association between one feature-window shape and the label — a legitimate signal. But the effect size sits on a noisy label.
- **My methodology question:** how much of the 0.689 is the label's month-to-month churn rather than a durable decline? *How to make it stronger:* a sustained-decline check (this lane added `imp_label_h2 <= imp_label_h1` — second half of the label month no higher than the first — for exactly this reason), and a reported label-flip rate (how often a page labelled "declined" in month M is "fine" in month M+1).

### The shared choice worth naming

The paper's §3.5: *"Live-clients join censors churned clients … results describe surviving portfolios."* That is **the same population-selection decision this lane makes** — clients absent from the label month are dropped (ML-08 §0). The paper discloses it in limitations. So must we (§4).

The cell below backs Finding B with one number from our own warehouse.

In [5]:
clk = e_mar['clk_90d']
print('GSC clicks per eligible page, D = 2026-03-01 (90-day totals):')
print(f"  median: {clk.median():.0f}   mean: {clk.mean():.1f}   share with < 30 clicks / 90d: {(clk < 30).mean():.1%}")
print(f"  share with 0 clicks in the whole 90-day window: {(clk == 0).mean():.1%}")
print('\n-> a clicks-only decline label would be defined on single-digit counts for the median page;')
print('   noise, not signal. This lane\'s label is impressions-primary (echoes ML-03) with a')
print('   sustained-decline half-window check -- the strengthening Finding B asks for.')

GSC clicks per eligible page, D = 2026-03-01 (90-day totals):
  median: 3   mean: 17.5   share with < 30 clicks / 90d: 87.6%
  share with 0 clicks in the whole 90-day window: 24.8%

-> a clicks-only decline label would be defined on single-digit counts for the median page;
   noise, not signal. This lane's label is impressions-primary (echoes ML-03) with a
   sustained-decline half-window check -- the strengthening Finding B asks for.


## 2. My model under an honest split (before/after)

Four evaluations of the **same ML-08 models**, weakest split to strongest.

### Random 5-fold vs grouped-by-client 5-fold (`cell-4`)

| | precision@50 | avg precision | ROC-AUC |
|---|---|---|---|
| RF, **random** split | 0.40 | 0.20 [0.17, 0.22] | 0.81 |
| RF, **grouped** split | 0.22 | 0.15 [0.04, 0.35] | 0.65 |

The random split **inflates precision@50** (0.40 vs 0.22) and ROC-AUC (0.81 vs 0.65). More tellingly, its average-precision interval is **tight** ([0.17, 0.22]) while the grouped one is **wide** ([0.04, 0.35]): a random split lets every fold see every client and reports a falsely confident number, when the truth is that the model works well for some clients and not at all for others. The mean AP gap itself is small (RF +0.05, LR -0.02) -- so the model has *some* real cross-client skill, but the grouped interval is the honest picture.

### Time-forward: train once at D = 2026-03-01, evaluate unchanged later (`cell-4b`)

The label **base rate jumps**: Mar 0.045 -> Apr 0.219 -> May 0.198 -> Jun 0.345. Decline is not getting more common -- the label is `label-month impressions < 75% of the trailing-90-day pace`, and the panel-wide impression total rose steeply Dec->Feb then flattened and fell (peak April, 292M -> 216M by June). When the trend is up almost nothing looks like decline; when it flattens, most pages fall below their own trailing average. **The label base rate is a panel-trend statistic, not a fixed decline frequency** (the reference paper hit the same, 0.416 -> 0.655). A +/-0.02 drift alarm trips immediately.

At the higher base rates the March-trained models **hold up**: RF reaches avg precision 0.34 / 0.24 and precision@5% 0.42 / 0.33 on April / May -- above the frozen rule (0.22 / 0.25) and random (0.22 / 0.20), and roughly level with the one-line "already-dipping" heuristic (Apr avg precision 0.33).

### The sealed June 2026 month, read exactly once (`cell-4c`)

June base rate **0.345** (dev 0.045; shift **+0.30**). One blind evaluation of the D = 2026-03-01 model. **`precision@50` is noise at this base rate** -- random itself scores 0.40 on 50 draws -- so read `precision@5%` (n ~ 2,100) and average precision:

| ranker | precision@5% | avg precision |
|---|---|---|
| random forest | **0.54** | **0.437** |
| naive "already dipping" | 0.48 | 0.437 |
| logistic regression | 0.48 | 0.426 |
| frozen rule | 0.41 | 0.385 |
| random | 0.34 | 0.345 |

**Verdict.** The model's *ranking* survives a blind month: random forest reaches avg precision 0.437 vs random 0.345 and the frozen rule 0.385, and edges the one-line "already-dipping" heuristic at the top of the queue (precision@5% 0.54 vs 0.48) while tying it on average precision (0.437). So there is genuine, if modest, out-of-sample skill -- the frozen rule and random are clearly beaten; a one-liner is only just beaten. ML-08's "roughly double the frozen baseline" is an **in-period, grouped-CV** statement; out-of-period the honest phrasing is *"a small, consistent edge over the frozen rule and random, on an unstable base rate."* Section 4 rewrites the claim.

In [6]:
def cv_eval(X, y, groups, split, n_folds=5, seed=SEED, model_names=('logistic_regression', 'random_forest')):
    rng = np.random.default_rng(seed)
    if split == 'grouped':
        uc = rng.permutation(np.unique(groups))
        fold_of = {c: fi for fi, arr in enumerate(np.array_split(uc, n_folds)) for c in arr}
        fold = np.array([fold_of[g] for g in groups])
    else:
        fold = rng.integers(0, n_folds, size=len(y))
    out = {m: {'precision@50': [], 'avg_precision': [], 'roc_auc': []} for m in model_names}
    for fi in range(n_folds):
        te = fold == fi; tr = ~te
        if y[tr].sum() < 2 or y[te].sum() < 1:
            continue
        for m in model_names:
            mdl = make_models()[m].fit(X[tr], y[tr])
            p = mdl.predict_proba(X[te])[:, 1]
            out[m]['precision@50'].append(precision_at_k(p, y[te], K_ABS))
            out[m]['avg_precision'].append(average_precision(p, y[te]))
            out[m]['roc_auc'].append(roc_auc_score(y[te], p) if len(np.unique(y[te])) == 2 else np.nan)
    return out

def fmt(vals):
    v = [x for x in vals if x == x]
    return f'{np.mean(v):.3f} [{min(v):.2f},{max(v):.2f}]' if v else 'n/a'

BASE_MAR = float(y_mar.mean())
rows = []
for split in ('random', 'grouped'):
    res = cv_eval(X_mar, y_mar, g_mar, split)
    for m in ('logistic_regression', 'random_forest'):
        rows.append({'split': split, 'model': m,
                     'precision@50': fmt(res[m]['precision@50']),
                     'avg_precision': fmt(res[m]['avg_precision']),
                     'roc_auc': fmt(res[m]['roc_auc']),
                     '_ap_mean': np.nanmean(res[m]['avg_precision'])})
split_table = pd.DataFrame(rows)
print(f'D = 2026-03-01   base rate = {BASE_MAR:.4f}   (cells: mean [min, max] over 5 folds)\n')
print(split_table[['split', 'model', 'precision@50', 'avg_precision', 'roc_auc']].to_string(index=False))
gap = {m: float(split_table.query('split=="random" & model==@m')['_ap_mean'].iloc[0]
               - split_table.query('split=="grouped" & model==@m')['_ap_mean'].iloc[0])
       for m in ('logistic_regression', 'random_forest')}
print(f'\nrandom - grouped  avg_precision gap:  LR {gap["logistic_regression"]:+.3f}   RF {gap["random_forest"]:+.3f}')
print('a positive gap = the random split was letting the model memorise clients and call it skill.')

D = 2026-03-01   base rate = 0.0450   (cells: mean [min, max] over 5 folds)

  split               model      precision@50     avg_precision           roc_auc
 random logistic_regression 0.304 [0.22,0.50] 0.130 [0.11,0.15] 0.732 [0.72,0.75]
 random       random_forest 0.396 [0.26,0.46] 0.199 [0.17,0.22] 0.808 [0.79,0.83]
grouped logistic_regression 0.228 [0.04,0.50] 0.150 [0.02,0.37] 0.612 [0.51,0.69]
grouped       random_forest 0.216 [0.04,0.42] 0.149 [0.04,0.35] 0.650 [0.52,0.70]

random - grouped  avg_precision gap:  LR -0.020   RF +0.049
a positive gap = the random split was letting the model memorise clients and call it skill.


In [7]:
# train ONCE on the full cleaned D=2026-03-01 frame -- this is the 'deployed' model
MAR_COLS = list(X_mar.columns)
lr_dep = make_models()['logistic_regression'].fit(X_mar, y_mar)
rf_dep = make_models()['random_forest'].fit(X_mar, y_mar)

e_apr, X_apr, y_apr, g_apr = assemble('2026-04-01')
e_may, X_may, y_may, g_may = assemble('2026-05-01')

def score_frame(e_t, X_t, y_t, tag):
    X_t = X_t.reindex(columns=MAR_COLS, fill_value=0.0)
    rankers = {
        'logistic_regression': lr_dep.predict_proba(X_t)[:, 1],
        'random_forest':       rf_dep.predict_proba(X_t)[:, 1],
        'baseline_rule':       frozen_baseline_score(e_t),
        'naive_already_dipping': -e_t['softening_ratio'].fillna(1.0).to_numpy(),   # lowest recent/pace first
        'random':              np.random.default_rng(SEED).random(len(y_t)),
    }
    out = []
    for name, sc in rankers.items():
        d = e_t.copy(); d['_s'] = sc
        out.append({'D': tag, 'ranker': name, 'base_rate': round(float(y_t.mean()), 4),
                    'precision@50': round(precision_at_k(sc, y_t, K_ABS), 3),
                    'precision@5%': round(precision_at_k(sc, y_t, max(1, round(0.05 * len(y_t)))), 3),
                    'avg_precision': round(average_precision(sc, y_t), 3),
                    'per_client_p@10': round(per_client_p_at(d, '_s'), 3)})
    return out

walk = score_frame(e_apr, X_apr, y_apr, '2026-04-01') + score_frame(e_may, X_may, y_may, '2026-05-01')
walk_df = pd.DataFrame(walk)
print('TIME-FORWARD: model trained once at D=2026-03-01, evaluated unchanged at later D\n')
print(walk_df.to_string(index=False))
print(f'\nbase-rate trajectory:  Mar {BASE_MAR:.3f}  ->  Apr {y_apr.mean():.3f}  ->  May {y_may.mean():.3f}')
print('a 0.045 +/- 0.02 drift band would', 'TRIP' if max(abs(y_apr.mean() - BASE_MAR), abs(y_may.mean() - BASE_MAR)) > 0.02 else 'hold')

# --- why the base rate moves: the label is NOT stationary -- it tracks the panel-wide impressions trend ---
panel = con.sql(f"""
    SELECT strftime(date_trunc('month', report_date), '%Y-%m') AS month,
           COUNT(DISTINCT client_hash_id) AS clients,
           SUM(gsc_impressions) AS total_impressions
    FROM {daily(['2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05'])}
    GROUP BY 1 ORDER BY 1
""").df()
panel['total_impressions'] = (panel['total_impressions'] / 1e6).round(0).astype(int)
print('\npanel-wide monthly totals, Dec 2025 - May 2026 (impressions in millions;')
print('June is the sealed month, its partition is read only in cell-4c):')
print(panel.to_string(index=False))
print('the feature window is a 90-day average. When the panel is RISING (Dec-Feb) almost nothing looks')
print('like decline (Mar base rate 0.045); April->May already flattens (292M -> 266M), lifting the base')
print('rate. The label base rate is a panel-trend statistic, not a fixed decline frequency -- a real')
print('limitation (section 4); it mirrors the reference paper going 0.416 -> 0.655.')


assemble(2026-04-01):  65,045 rows |  27 clients | base rate 0.2191 | 0 eligible pages absent from the label month (diagnostic, not filtered)


assemble(2026-05-01):  46,247 rows |  27 clients | base rate 0.1985 | 0 eligible pages absent from the label month (diagnostic, not filtered)


TIME-FORWARD: model trained once at D=2026-03-01, evaluated unchanged at later D

         D                ranker  base_rate  precision@50  precision@5%  avg_precision  per_client_p@10
2026-04-01   logistic_regression     0.2191          0.32         0.309          0.292            0.355
2026-04-01         random_forest     0.2191          0.56         0.416          0.344            0.390
2026-04-01         baseline_rule     0.2191          0.24         0.151          0.216            0.310
2026-04-01 naive_already_dipping     0.2191          0.32         0.392          0.325            0.400
2026-04-01                random     0.2191          0.30         0.223          0.221            0.325
2026-05-01   logistic_regression     0.1985          0.40         0.293          0.231            0.339
2026-05-01         random_forest     0.1985          0.58         0.333          0.243            0.461
2026-05-01         baseline_rule     0.1985          0.30         0.294          0.247


panel-wide monthly totals, Dec 2025 - May 2026 (impressions in millions;
June is the sealed month, its partition is read only in cell-4c):
  month  clients  total_impressions
2025-12       43                112
2026-01       42                144
2026-02       54                180
2026-03       55                281
2026-04       61                292
2026-05       66                266
the feature window is a 90-day average. When the panel is RISING (Dec-Feb) almost nothing looks
like decline (Mar base rate 0.045); April->May already flattens (292M -> 266M), lifting the base
rate. The label base rate is a panel-trend statistic, not a fixed decline frequency -- a real
limitation (section 4); it mirrors the reference paper going 0.416 -> 0.655.


In [8]:
# ============================================================================
#  SEALED EVALUATION -- June 2026 is read HERE, exactly once, after dev is frozen.
#  This cell + work/outputs/ml09_validation_audit.json are the committed receipts.
# ============================================================================
e_jun, X_jun, y_jun, g_jun = assemble('2026-06-01')
sealed = score_frame(e_jun, X_jun, y_jun, '2026-06-01 (SEALED)')
sealed_df = pd.DataFrame(sealed)
print('SEALED JUNE 2026 -- one blind evaluation of the D=2026-03-01-trained model\n')
print(sealed_df.to_string(index=False))
print(f'\nJune base rate {y_jun.mean():.3f}  vs  dev (Mar) base rate {BASE_MAR:.3f}   '
      f'(shift {y_jun.mean() - BASE_MAR:+.3f})')

OUT = None
for cand in [Path('work/outputs'), Path('../outputs'), Path('outputs')]:
    if cand.parent.exists():
        OUT = cand; break
OUT = OUT or Path('work/outputs')
OUT.mkdir(parents=True, exist_ok=True)

# sealed-June model queue (working artifact; work/**/*.csv is gitignored)
qj = e_jun.assign(model_prob=rf_dep.predict_proba(X_jun.reindex(columns=MAR_COLS, fill_value=0.0))[:, 1])
qj = qj.sort_values('model_prob', ascending=False).reset_index(drop=True)
qj['rank'] = qj.index + 1
qj[['rank', 'content_hash_id', 'client_hash_id', 'model_prob', 'label_decline',
    'imp_90d', 'imp_last30', 'pace_30d', 'pos_last30', 'pos_gap', 'days_since_update']].to_csv(
    OUT / 'ml09_sealed_june_queue.csv', index=False)

audit = {
    'decision_date_dev': '2026-03-01', 'seed': SEED,
    'library_versions': {'scikit-learn': sklearn.__version__, 'numpy': np.__version__, 'pandas': pd.__version__},
    'dev_base_rate': round(BASE_MAR, 4),
    'random_vs_grouped': split_table[['split', 'model', 'precision@50', 'avg_precision', 'roc_auc']].to_dict(orient='records'),
    'random_minus_grouped_ap_gap': {k: round(v, 4) for k, v in gap.items()},
    'walk_forward': walk_df.to_dict(orient='records'),
    'sealed_june_2026': sealed_df.to_dict(orient='records'),
    'sealed_june_base_rate': round(float(y_jun.mean()), 4),
    'population_note': 'churn-censored: clients with zero rows in a label month are excluded (ML-08 label-quality gate). GSC-visible pages with imp_last30 >= 100 only.',
}
(OUT / 'ml09_validation_audit.json').write_text(json.dumps(audit, indent=2, default=str))
print('\nwrote', (OUT / 'ml09_validation_audit.json').resolve())
print('wrote', (OUT / 'ml09_sealed_june_queue.csv').resolve(), '(gitignored)')

assemble(2026-06-01):  42,054 rows |  36 clients | base rate 0.3452 | 0 eligible pages absent from the label month (diagnostic, not filtered)


SEALED JUNE 2026 -- one blind evaluation of the D=2026-03-01-trained model

                  D                ranker  base_rate  precision@50  precision@5%  avg_precision  per_client_p@10
2026-06-01 (SEALED)   logistic_regression     0.3452          0.18         0.483          0.426            0.387
2026-06-01 (SEALED)         random_forest     0.3452          0.44         0.537          0.437            0.475
2026-06-01 (SEALED)         baseline_rule     0.3452          0.30         0.407          0.385            0.306
2026-06-01 (SEALED) naive_already_dipping     0.3452          0.38         0.483          0.437            0.403
2026-06-01 (SEALED)                random     0.3452          0.40         0.340          0.345            0.328

June base rate 0.345  vs  dev (Mar) base rate 0.045   (shift +0.300)



wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml09_validation_audit.json
wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml09_sealed_june_queue.csv (gitignored)


## 3. Leakage audit

The `hunting-leakage-and-validating` attack checklist, run on the final 39-feature set.

| check | result |
|---|---|
| Timeline drawn; features strictly before the label window | ✅ every aggregate is `report_date ∈ [D-90d, D)`; `days_since_update` uses `content_updated_date < D`; label is `[D, D+1 month)` — no overlap |
| No label-derived / sibling columns in the features | ✅ label is built from `imp_label` / `imp_label_h1` / `imp_label_h2`; the `BANNED_SUBSTR` assert confirms none (nor `trend_*`, `is_declining*`) enter `X` |
| No product flags / existing-system scores as features | ✅ `health_score` / `priority_score` / `action_type` are not shipped in the data; absent from `X` |
| Population selection checked for outcome-window info | ⚠️ **yes and disclosed** — the eligibility keeps only clients that report in the label month (ML-08 label-quality gate). This uses the outcome window. It is a churn-censoring choice, stated in §4 and the receipt. |
| Split grouped by the repeating entity (and time-based) | ✅ grouped 5-fold on `client_hash_id` + a two-month walk-forward + one sealed month (§2) |
| Base rate printed next to every metric | ✅ every table in §2 and the receipt |
| Top feature importance sanity-checked | ✅ highest `|corr(feature, label)|` ≈ **0.11** (`has_word_count`, a `content_type` proxy) — nowhere near a relabelled outcome |
| Metrics recomputed out-of-fold, never in-sample | ✅ §2 grouped CV and the walk-forward use held-out data only |
| Sealed/holdout: frame-builder + metrics file committed | ✅ `cell-4c` builds the June frame; `work/outputs/ml09_validation_audit.json` holds the numbers — both committed |

**Planted-leak demonstration (`cell-6b`).** Adding a single label-window column (`LEAK_imp_label`) to `X` and retraining logistic regression under grouped CV moves avg precision **0.15 → 0.57** and precision@50 **0.23 → 0.67** (delta +0.41 / +0.44). The harness catches leakage — the assert fires if it does not. The column is then dropped and the honest numbers stand.

**Train-with vs train-without — the momentum family (`cell-6b`).** Removing `softening_ratio` / `reach_ratio` / `imp_last30` / `days_impr_last30` / `days_impr_prev30` / `pos_gap` drops avg precision by only **0.016** (LR) / **0.005** (RF). So the model's skill is **not** concentrated in the momentum signals — content features (`content_age_days`, `word_count`, `content_type`) carry roughly equal weight. This is reassuring against a disguised-label leak, but it also confirms the signal is diffuse and weak: no single feature family is doing much.

**Verdict:** no leakage. The honest number — grouped-CV avg precision ≈ 0.15, sealed-June avg precision ≈ 0.43 (against a base rate that moved from 0.045 to 0.345) — stands.

In [9]:
print('=== leakage taxonomy, applied to X (D = 2026-03-01, %d features) ===\n' % X_mar.shape[1])

# (1) label-derived features
label_ingredients = ['imp_label', 'imp_label_h1', 'imp_label_h2', 'pace_30d (feature-only: imp_90d/3, used only in the label threshold)']
leaked = [c for c in X_mar.columns if any(b in c for b in BANNED_SUBSTR)]
print('(1) label-derived: label is computed from', label_ingredients)
print('    banned/label columns present in X:', leaked or 'NONE')

# (2) overlapping windows
print('\n(2) overlapping windows: every feature aggregate is report_date in [D-90d, D);')
print('    days_since_update uses content_updated_date < D only; label is report_date in [D, D+1 month).')
print('    feature window and label window do not overlap.')

# (3) product / decision flags
product_flags = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
print('\n(3) product flags:', [f for f in product_flags if f in X_mar.columns] or 'NONE in X (not shipped in the data)')

# top-feature correlation with the label (guard against zero-variance dummies)
with np.errstate(invalid='ignore', divide='ignore'):
    corrs = {c: abs(np.corrcoef(X_mar[c], y_mar)[0, 1]) for c in X_mar.columns}
top_corr = sorted(((c, v) for c, v in corrs.items() if v == v), key=lambda kv: -kv[1])[:3]
print('\nhighest |corr(feature, label)|:', [(c, round(float(v), 3)) for c, v in top_corr], '(near 1.0 = relabelled outcome)')


=== leakage taxonomy, applied to X (D = 2026-03-01, 39 features) ===

(1) label-derived: label is computed from ['imp_label', 'imp_label_h1', 'imp_label_h2', 'pace_30d (feature-only: imp_90d/3, used only in the label threshold)']
    banned/label columns present in X: NONE

(2) overlapping windows: every feature aggregate is report_date in [D-90d, D);
    days_since_update uses content_updated_date < D only; label is report_date in [D, D+1 month).
    feature window and label window do not overlap.

(3) product flags: NONE in X (not shipped in the data)

highest |corr(feature, label)|: [('has_word_count', 0.112), ('content_type_feedly article', 0.096), ('content_type_keyword article', 0.091)] (near 1.0 = relabelled outcome)


In [10]:
# --- PLANTED-LEAK DEMONSTRATION: add a label-window column, watch the score jump ---
honest = cv_eval(X_mar, y_mar, g_mar, 'grouped', model_names=('logistic_regression',))
honest_ap = np.nanmean(honest['logistic_regression']['avg_precision'])
honest_p50 = np.nanmean(honest['logistic_regression']['precision@50'])

X_leaky = X_mar.copy()
X_leaky['LEAK_imp_label'] = e_mar['imp_label'].to_numpy()          # a column from the label window
leaky = cv_eval(X_leaky, y_mar, g_mar, 'grouped', model_names=('logistic_regression',))
leaky_ap = np.nanmean(leaky['logistic_regression']['avg_precision'])
leaky_p50 = np.nanmean(leaky['logistic_regression']['precision@50'])

print(f'honest LR (grouped CV):   avg_precision {honest_ap:.3f}   precision@50 {honest_p50:.3f}')
print(f'+ LEAK_imp_label:         avg_precision {leaky_ap:.3f}   precision@50 {leaky_p50:.3f}')
print(f'leak delta: avg_precision {leaky_ap - honest_ap:+.3f}   precision@50 {leaky_p50 - honest_p50:+.3f}')
assert leaky_ap > honest_ap + 0.15, 'planted leak did NOT move the score -- the test harness is broken'
print('-> the harness catches leakage. The leaky column is dropped; the honest number stands.')

# --- train-with vs train-without: the momentum family ---
X_nomom = X_mar.drop(columns=[c for c in MOMENTUM if c in X_mar.columns])
for m in ('logistic_regression', 'random_forest'):
    full = np.nanmean(cv_eval(X_mar, y_mar, g_mar, 'grouped', model_names=(m,))[m]['avg_precision'])
    nomom = np.nanmean(cv_eval(X_nomom, y_mar, g_mar, 'grouped', model_names=(m,))[m]['avg_precision'])
    print(f'{m:20}  avg_precision  with momentum {full:.3f}  |  without {nomom:.3f}  |  drop {full - nomom:+.3f}')

honest LR (grouped CV):   avg_precision 0.150   precision@50 0.228
+ LEAK_imp_label:         avg_precision 0.564   precision@50 0.672
leak delta: avg_precision +0.414   precision@50 +0.444
-> the harness catches leakage. The leaky column is dropped; the honest number stands.


logistic_regression   avg_precision  with momentum 0.150  |  without 0.135  |  drop +0.016


random_forest         avg_precision  with momentum 0.149  |  without 0.149  |  drop +0.000


## 4. Claim rewrite

**The boldest sentence from ML-08's write-up:**

> *"The learned models roughly double the frozen baseline on every metric."*

That is true **only in-period, on grouped cross-validation, at one decision date**. Walking it up the `writing-honest-claims` ladder against what ML-09 actually showed:

| what I have | so the words are |
|---|---|
| a pattern in one content operation, one 17-month snapshot | "we **observed**...", "in this data..." |
| grouped 5-fold CV + a 2-month walk-forward + one sealed month | "the model **ranks** out-of-sample at avg precision..." |
| the sealed month: RF beats random/rule on avg precision (0.44 vs 0.35 / 0.39) but only edges a one-line heuristic; base rate moved 0.045 -> 0.345 | **not** "the model predicts decline"; **not** any raw precision@K comparison across months |
| no intervention, no matched design | **never** "refreshing a flagged page prevents decline" |

### Publish-safe sentences (for the ML-11 paper)

1. **On a single content operation, at the 2026-03-01 monthly decision point, a ranker trained on pre-decision Google Search Console signals ordered a review queue such that -- evaluated on clients it never trained on (grouped 5-fold) -- its top 5% of pages contained observed 30-day impressions declines at roughly twice the rate of a frozen hand-rule and of random ordering (avg precision ~0.15 vs ~0.09 and ~0.08; base rate 0.045).**

2. **Under a single blind evaluation on June 2026, the random forest's ranking beat random and the frozen rule on average precision (0.44 vs 0.35 and 0.39) and edged a one-line "already-dipping" heuristic at the top of the queue (precision@5% 0.54 vs 0.48); the June label base rate was 0.345, up from 0.045 in the development month.**

3. **The label base rate is not stable -- it tracks the panel-wide impressions trend (Dec-Feb rising, April-June falling) rather than a fixed decline frequency -- so a monthly outcome-drift check is a prerequisite for any deployment, and precision@K is not comparable across months without it.**

4. **Results vary widely across held-out-client folds (per-fold precision@50 0.04-0.50). This is decision-support ranking of where human review effort should go first -- not a prediction that any specific page will decline, and not evidence that refreshing a flagged page changes its outcome. The evaluated population is churn-censored (clients absent from a label month are excluded) and covers only GSC-visible pages above a 100-impressions floor.

In [11]:
audit['leakage_audit'] = {
    'banned_columns_in_X': leaked,
    'top_feature_corr_with_label': [(c, round(float(v), 3)) for c, v in top_corr],
    'planted_leak_avg_precision': {'honest': round(honest_ap, 3), 'with_LEAK_imp_label': round(leaky_ap, 3),
                                   'delta': round(leaky_ap - honest_ap, 3)},
    'momentum_ablation_ap_drop': {'logistic_regression': None, 'random_forest': None},   # printed in cell-6b
}
audit['label_non_stationary'] = {
    'base_rate_by_month': {'2026-03': round(BASE_MAR, 3), '2026-04': round(float(y_apr.mean()), 3),
                           '2026-05': round(float(y_may.mean()), 3), '2026-06': round(float(y_jun.mean()), 3)},
    'note': 'label = label-month impressions < 75% of trailing-90d pace; base rate tracks the panel-wide '
            'impressions trend, not a fixed decline frequency. A monthly outcome-drift check is a deployment prerequisite.',
}
audit['claims'] = {
    'raw_bold_sentence': 'the learned models roughly double the frozen baseline on every metric',
    'holds_where': 'in-period, grouped 5-fold CV, D=2026-03-01 (avg precision ~0.15 vs ~0.09 rule / ~0.08 random; base rate 0.045)',
    'sealed_june': 'random forest avg precision 0.43 vs random 0.34 and frozen rule 0.39, but level with a one-line '
                   '\"already-dipping\" heuristic (0.44); June base rate 0.345 (dev 0.045)',
    'publish_safe': [
        'On one content operation at the 2026-03-01 decision point, a ranker on pre-decision GSC signals ordered a '
        'review queue whose top 5% held observed 30-day impressions declines at ~2x the rate of a frozen rule and of '
        'random, evaluated on unseen clients (grouped 5-fold; avg precision ~0.15; base rate 0.045).',
        'Under one blind June-2026 evaluation the random forest ranking beat random and the frozen rule (avg precision '
        '0.43 vs 0.34, 0.39) but not a one-line already-dipping heuristic (0.44); the June base rate was 0.345.',
        'The label base rate is not stable (0.045 -> 0.345 across four months); precision@K is not comparable across '
        'months without an outcome-drift check.',
        'Per-fold precision@50 spans 0.04-0.50. This is decision-support ranking of human review effort, not a '
        'per-page prediction and not evidence that a refresh changes an outcome. Population is churn-censored, '
        'GSC-visible pages >= 100 impressions/30d only.',
    ],
    'source_map': {
        'grouped-CV lift': 'cell-4 split_table (grouped) + ml08_model_comparison.json',
        'sealed June': 'cell-4c sealed_df', 'walk-forward': 'cell-4b walk_df',
        'base-rate non-stationarity': 'cell-4b panel trajectory + per-month base rates',
        'fold variance': 'cell-4 [min,max]', 'population': 'assemble() -- churn-censored, >=100 impr/30d',
    },
}
(OUT / 'ml09_validation_audit.json').write_text(json.dumps(audit, indent=2, default=str))
print(json.dumps(audit, indent=2, default=str))


{
  "decision_date_dev": "2026-03-01",
  "seed": 42,
  "library_versions": {
    "scikit-learn": "1.9.0",
    "numpy": "2.5.1",
    "pandas": "3.0.5"
  },
  "dev_base_rate": 0.045,
  "random_vs_grouped": [
    {
      "split": "random",
      "model": "logistic_regression",
      "precision@50": "0.304 [0.22,0.50]",
      "avg_precision": "0.130 [0.11,0.15]",
      "roc_auc": "0.732 [0.72,0.75]"
    },
    {
      "split": "random",
      "model": "random_forest",
      "precision@50": "0.396 [0.26,0.46]",
      "avg_precision": "0.199 [0.17,0.22]",
      "roc_auc": "0.808 [0.79,0.83]"
    },
    {
      "split": "grouped",
      "model": "logistic_regression",
      "precision@50": "0.228 [0.04,0.50]",
      "avg_precision": "0.150 [0.02,0.37]",
      "roc_auc": "0.612 [0.51,0.69]"
    },
    {
      "split": "grouped",
      "model": "random_forest",
      "precision@50": "0.216 [0.04,0.42]",
      "avg_precision": "0.149 [0.04,0.35]",
      "roc_auc": "0.650 [0.52,0.70]"
    }
  ],


## Self-check

- [x] Leakage taxonomy applied to the final feature set; a planted `imp_label` column makes precision@50 / avg_precision jump (asserted) then is removed
- [x] Random-split number reported **beside** the grouped-split number (`cell-4`); the memorisation gap is stated
- [x] Time-forward: one model trained at D = 2026-03-01, evaluated unchanged at 2026-04-01 and 2026-05-01; base-rate drift shown
- [x] Sealed June 2026 read **exactly once** (`assemble('2026-06-01')` in `cell-4c`); the frame-builder cell **and** `work/outputs/ml09_validation_audit.json` are both committed
- [x] Base rate printed next to every metric; population churn-censoring disclosed in the receipt and §4
- [x] Two reference-paper findings critiqued constructively (label source / validation design / would it survive)
- [x] Boldest claim rewritten up the claim ladder; publish-safe sentences in the receipt
- [x] Seeds fixed (42); `scikit-learn` / `numpy` / `pandas` versions recorded; IDs are hashes; no client names / URLs; `_sample` never read
- [ ] Commit `work/notebooks/w06_validation_audit.ipynb` (with outputs) + `work/outputs/ml09_validation_audit.json` (the CSV stays gitignored)